# 14 · Modelo estrella e indicadores (KPI) para el tablero

**Propósito:** dejar en gold las tablas que lee el tablero del CCO, en un **modelo estrella**: una tabla de hechos
con una fila por salida programada de LATAM y dimensiones pequeñas para filtrar y agrupar. Encima se calculan los
indicadores diarios y mensuales.

| Tabla | Grano | Para qué |
|---|---|---|
| `fact_vuelo` | una salida programada de LATAM (incluye canceladas) | demora real, alerta emitida, clima y red a T−2 h |
| `dim_fecha` | un día | año, mes, día de la semana, fin de semana |
| `dim_aeropuerto` | un aeropuerto | nombre, país, coordenadas, zona horaria, si es hub, si tiene METAR |
| `dim_aerolinea` | una aerolínea | nombre, grupo (LATAM o competencia), pasajeros o carga |
| `dim_ruta` | origen-destino | vuelos, si es internacional |
| `dim_condicion_clima` | categoría de vuelo | VFR, MVFR, IFR, LIFR o sin dato |
| `kpi_diario` | día × aeropuerto de origen | K1, K2 y K3 |
| `kpi_alerta_mensual` | mes | K4 |

**Entrada:** `silver/vuelos`, `gold/features_salida`, `gold/predicciones` y `bronze/ref/aeropuertos`.

**Salida:** las tablas de arriba en gold, el catálogo de Glue actualizado (para Athena) y una copia de los KPI en
`docs/evidencias/tablero/`.

**Requisito:** notebooks 05, 08 y 13.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import time

import matplotlib.pyplot as plt

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-estrella-kpi")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
EVID = RAIZ / "docs" / "evidencias" / "tablero"
EVID.mkdir(parents=True, exist_ok=True)
UMBRAL_AMBAR = cfg["umbrales_demora_min"]["ambar"]
PREFIJOS_BRASIL = ("SB", "SD", "SI", "SJ", "SN", "SS", "SW")

vuelos = en_periodo(spark.read.parquet(ruta("silver_vuelos"))).where(F.col("es_latam"))
features = spark.read.parquet(ruta("gold_features")).select(
    "id_vuelo", "o_categoria_vuelo", "red_frac_atrasadas", "red_demora_media", "tasa_hist_origen_hora")
predicciones = spark.read.parquet(ruta("gold_predicciones")).select(
    "id_vuelo", "prob_demora15", "prob_demora60", "nivel_alerta", "version_modelo", "particion")

## 1. Tabla de hechos `fact_vuelo`

Las claves apuntan a las dimensiones y `id_vuelo` permite seguir cualquier cifra del tablero hasta su fila en silver
y, desde ahí, hasta el archivo de la ANAC (linaje). Las salidas canceladas o fuera del modelo quedan con
`nivel_alerta = sin_prediccion`.

In [ ]:
fact = (
    vuelos.join(features, "id_vuelo", "left").join(predicciones, "id_vuelo", "left")
    .select(
        "id_vuelo",
        F.date_format("fecha_local", "yyyyMMdd").cast("int").alias("fecha_key"),
        F.col("origen").alias("origen_key"),
        F.col("destino").alias("destino_key"),
        F.col("aerolinea").alias("aerolinea_key"),
        F.concat_ws("-", "origen", "destino").alias("ruta_key"),
        F.coalesce("o_categoria_vuelo", F.lit("SIN_DATO")).alias("condicion_clima_key"),
        "pais", "numero_vuelo", "salida_prog_utc", "fecha_local",
        F.col("hora_local_salida").alias("hora_prog_local"),
        F.col("demora_salida_min").cast("double").alias("demora_salida_min"),
        F.col("y_demora15").cast("int").alias("es_demora15"),
        F.col("y_demora60").cast("int").alias("es_demora60"),
        F.col("cancelado").cast("int").alias("cancelado"),
        F.col("desviado").cast("int").alias("desviado"),
        F.col("es_atipico").cast("int").alias("es_atipico"),
        "red_frac_atrasadas", "red_demora_media", "tasa_hist_origen_hora",
        "prob_demora15", "prob_demora60",
        F.coalesce("nivel_alerta", F.lit("sin_prediccion")).alias("nivel_alerta"),
        "version_modelo", "particion", "year", "month",
    )
)
fact.repartition("year", "month").write.mode("overwrite").partitionBy("year", "month").parquet(ruta("gold_fact_vuelo"))
fact = spark.read.parquet(ruta("gold_fact_vuelo"))
print(f"fact_vuelo: {fact.count():,} salidas programadas de LATAM")

## 2. Dimensiones

Son pequeñas: se reconstruyen completas en cada corrida.

In [ ]:
def escribir_dimension(df, tabla):
    df.coalesce(1).write.mode("overwrite").parquet(ruta(tabla))
    print(f"{tabla}: {spark.read.parquet(ruta(tabla)).count():,} filas")


# dim_fecha: todos los días del periodo, aunque no haya vuelos
rango = fact.agg(F.min("fecha_local").alias("a"), F.max("fecha_local").alias("b"))
dim_fecha = (
    rango.select(F.explode(F.sequence("a", "b")).alias("fecha"))
    .withColumn("fecha_key", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha")).withColumn("mes", F.month("fecha")).withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("trimestre", F.quarter("fecha")).withColumn("dia_semana", F.dayofweek("fecha"))
    .withColumn("es_fin_de_semana", F.col("dia_semana").isin(1, 7).cast("int"))
    .withColumn("anio_mes", F.date_format("fecha", "yyyy-MM"))
)
escribir_dimension(dim_fecha, "gold_dim_fecha")

# dim_aeropuerto: ficha de IEM (notebook 03) de los aeropuertos que aparecen en los hechos
esquema_fichas = T.StructType([
    T.StructField("aeropuerto", T.StringType()), T.StructField("estacion", T.StringType()),
    T.StructField("nombre", T.StringType()), T.StructField("pais", T.StringType()),
    T.StructField("latitud", T.DoubleType()), T.StructField("longitud", T.DoubleType()),
    T.StructField("zona_horaria", T.StringType()), T.StructField("tiene_metar", T.BooleanType()),
])
fichas = spark.read.option("header", True).schema(esquema_fichas).csv(f"{ruta('bronze_aeropuertos')}/aeropuertos_iem.csv")
hubs = cfg["modelo_final"]["hubs"]   # los mismos hubs que usa el tablero, definidos en config/pipeline.yaml
usados = fact.select(F.col("origen_key").alias("aeropuerto")).union(fact.select(F.col("destino_key").alias("aeropuerto"))).distinct()
dim_aeropuerto = (
    usados.join(fichas, "aeropuerto", "left")
    .withColumn("aeropuerto_key", F.col("aeropuerto"))
    .withColumn("es_brasil", F.substring("aeropuerto", 1, 2).isin(*PREFIJOS_BRASIL))
    .withColumn("es_hub_latam", F.col("aeropuerto").isin(hubs))
    .withColumn("tiene_metar", F.coalesce("tiene_metar", F.lit(False)))
)
escribir_dimension(dim_aeropuerto, "gold_dim_aeropuerto")

# dim_aerolinea: grupo LATAM (pasajeros y carga) y competidores, desde config/pipeline.yaml
aerolineas = (
    [(c, n, "LATAM Airlines Group", "pasajeros") for c, n in cfg["latam"]["pasajeros"].items()]
    + [(c, n, "LATAM Airlines Group", "carga") for c, n in cfg["latam"]["carga"].items()]
    + [(c, n, "Competidor en Brasil", "pasajeros") for c, n in cfg["competidores"].items()]
)
dim_aerolinea = spark.createDataFrame(aerolineas, "aerolinea_key string, nombre string, grupo string, tipo string")
escribir_dimension(dim_aerolinea, "gold_dim_aerolinea")

# dim_ruta
dim_ruta = (
    fact.groupBy("ruta_key", F.col("origen_key").alias("origen"), F.col("destino_key").alias("destino"))
    .agg(F.count("*").alias("salidas_programadas"))
    .withColumn("es_internacional", ~(F.substring("origen", 1, 2).isin(*PREFIJOS_BRASIL)
                                      & F.substring("destino", 1, 2).isin(*PREFIJOS_BRASIL)))
)
escribir_dimension(dim_ruta, "gold_dim_ruta")

# dim_condicion_clima: categorías de vuelo con las reglas del notebook 06
condiciones = [
    ("VFR", "Visual: techo > 3000 ft y visibilidad > 5 mi", 1),
    ("MVFR", "Visual marginal: techo 1000-3000 ft o visibilidad 3-5 mi", 2),
    ("IFR", "Instrumental: techo 500-999 ft o visibilidad 1 a menos de 3 mi", 3),
    ("LIFR", "Instrumental bajo: techo < 500 ft o visibilidad < 1 mi", 4),
    ("SIN_DATO", "Sin METAR vigente a T-2 h", 0),
]
escribir_dimension(spark.createDataFrame(condiciones, "condicion_clima_key string, descripcion string, severidad int"),
                   "gold_dim_condicion_clima")

## 3. Indicadores

LATAM no publica el organigrama de su CCO: la columna "Responsable" nombra el **rol propuesto** y hay que validarlo
con la empresa.

| KPI | Definición | Fórmula | Frecuencia | Responsable | Meta / lectura |
|---|---|---|---|---|---|
| **K1** Puntualidad de salida | Salidas operadas que salen con menos de 15 min de demora | salidas con demora < 15 / salidas operadas | diaria por aeropuerto; mensual en el tablero | Jefe de turno del CCO (rol propuesto) | Subir desde la línea base de 2025 (notebook 15) |
| **K2** Alertas emitidas | Salidas en alerta ámbar y roja a T−2 h | conteo por nivel | diaria por aeropuerto | Jefe de turno del CCO (rol propuesto) | Ver cuántos vuelos por día hay que reprogramar; la capacidad del CCO la debe confirmar LATAM |
| **K3** Índice de propagación | Qué tan atrasada viene la red del aeropuerto antes del vuelo | promedio de `red_frac_atrasadas` (salidas previas con ≥ 15 min en las 2 h antes de T−2 h) | diaria por aeropuerto | Coordinador de hub (rol propuesto) | Escalar cuando el aeropuerto entre en el tramo de red con mayor tasa de demora medida (hallazgos del notebook 15) |
| **K4** Calidad de la alerta | De las alertas, cuántas acertaron (precisión) y de las demoras, cuántas se avisaron (recall) | VP / (VP + FP) y VP / (VP + FN) | mensual | Analítica de operaciones (rol propuesto) | Reentrenar si la lista del modelo evita menos costo que la tabla de tasas (notebook 16) |

**Cobertura de K2 y K4:** las predicciones (notebook 13) existen solo para `valid` y `test`, es decir desde 2024-07. Antes de esa fecha K2 vale 0 porque no se calculó, no porque no haya alertas. En K4 la columna `particion` separa `valid` (con ella se eligió el umbral, así que sus cifras son optimistas) de `test` (2025), que es la evaluación sin sesgo.

In [ ]:
operada = (F.col("cancelado") == 0) & F.col("demora_salida_min").isNotNull()
kpi_diario = (
    fact.groupBy("fecha_local", "fecha_key", F.col("origen_key").alias("origen"))
    .agg(
        F.count("*").alias("salidas_programadas"),
        F.sum(operada.cast("int")).alias("salidas_operadas"),
        F.sum("cancelado").alias("cancelaciones"),
        F.sum((operada & (F.col("demora_salida_min") < UMBRAL_AMBAR)).cast("int")).alias("salidas_puntuales"),
        # la media excluye las demoras atípicas (es_atipico, fuera del rango de calidad de config/pipeline.yaml): una sola de 1440 min la distorsiona
        F.round(F.avg(F.when(operada & (F.col("es_atipico") == 0), F.col("demora_salida_min"))), 2).alias("demora_media_min"),
        F.sum((F.col("nivel_alerta") == "ambar").cast("int")).alias("k2_alertas_ambar"),
        F.sum((F.col("nivel_alerta") == "rojo").cast("int")).alias("k2_alertas_rojas"),
        F.round(F.avg("red_frac_atrasadas"), 4).alias("k3_indice_propagacion"),
    )
    .withColumn("k1_puntualidad", F.round(F.col("salidas_puntuales") / F.col("salidas_operadas"), 4))
    .withColumn("year", F.year("fecha_local")).withColumn("month", F.month("fecha_local"))
)
(
    kpi_diario.repartition("year", "month")
    .write.mode("overwrite").partitionBy("year", "month").parquet(ruta("gold_kpi_diario"))
)

# K4: alerta roja contra demora real de 60 min o más; ámbar o roja contra 15 min o más
con_pred = fact.where(F.col("nivel_alerta") != "sin_prediccion").where(F.col("es_demora15").isNotNull())
roja = F.col("nivel_alerta") == "rojo"
ambar_o_roja = F.col("nivel_alerta").isin("ambar", "rojo")


def suma(condicion):
    return F.sum(condicion.cast("int"))


kpi_mensual = (
    con_pred.groupBy(F.date_format("fecha_local", "yyyy-MM").alias("anio_mes"), "version_modelo", "particion")
    .agg(
        F.count("*").alias("vuelos"),
        suma(roja & (F.col("es_demora60") == 1)).alias("rojo_vp"), suma(roja & (F.col("es_demora60") == 0)).alias("rojo_fp"),
        suma(~roja & (F.col("es_demora60") == 1)).alias("rojo_fn"),
        suma(ambar_o_roja & (F.col("es_demora15") == 1)).alias("ambar_vp"),
        suma(ambar_o_roja & (F.col("es_demora15") == 0)).alias("ambar_fp"),
        suma(~ambar_o_roja & (F.col("es_demora15") == 1)).alias("ambar_fn"),
    )
    .withColumn("k4_precision_roja", F.round(F.col("rojo_vp") / (F.col("rojo_vp") + F.col("rojo_fp")), 4))
    .withColumn("k4_recall_roja", F.round(F.col("rojo_vp") / (F.col("rojo_vp") + F.col("rojo_fn")), 4))
    .withColumn("k4_precision_ambar", F.round(F.col("ambar_vp") / (F.col("ambar_vp") + F.col("ambar_fp")), 4))
    .withColumn("k4_recall_ambar", F.round(F.col("ambar_vp") / (F.col("ambar_vp") + F.col("ambar_fn")), 4))
    .orderBy("anio_mes")
)
kpi_mensual.coalesce(1).write.mode("overwrite").parquet(ruta("gold_kpi_alerta_mensual"))
kpi_mensual = spark.read.parquet(ruta("gold_kpi_alerta_mensual")).orderBy("anio_mes").toPandas()
kpi_mensual.to_csv(EVID / "kpi_alerta_mensual.csv", index=False)
kpi_mensual

## 4. Vista del tablero: puntualidad y alertas en los hubs

In [ ]:
AZUL, NARANJA, GRIS, TINTA = "#2a78d6", "#eb6834", "#c9c8c2", "#1f1f1f"
hubs_tablero = cfg["modelo_final"]["hubs"]
mensual = (
    spark.read.parquet(ruta("gold_kpi_diario")).where(F.col("origen").isin(hubs_tablero))
    .groupBy(F.date_format("fecha_local", "yyyy-MM").alias("anio_mes"), "origen")
    .agg(F.sum("salidas_puntuales").alias("p"), F.sum("salidas_operadas").alias("o"),
         F.sum("k2_alertas_ambar").alias("ambar"), F.sum("k2_alertas_rojas").alias("rojo"))
    .withColumn("k1_puntualidad", F.col("p") / F.col("o"))
    .orderBy("anio_mes").toPandas()
)
mensual.to_csv(EVID / "kpi_mensual_hubs.csv", index=False)

fig, (e1, e2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for hub, color in zip(hubs_tablero, (AZUL, NARANJA)):
    parte = mensual[mensual.origen == hub]
    e1.plot(parte.anio_mes, parte.k1_puntualidad * 100, color=color, marker="o", ms=3, label=hub)
    e2.bar(parte.anio_mes, parte.ambar + parte.rojo, color=color, alpha=0.7, label=hub)
e1.set_title("K1 · ¿Qué porcentaje de salidas de LATAM sale a tiempo en los hubs?", loc="left", fontsize=11, color=TINTA)
e1.set_ylabel("% salidas con < 15 min")
e2.set_title("K2 · ¿Cuántas alertas (ámbar + roja) recibió el CCO por mes?", loc="left", fontsize=11, color=TINTA)
for e in (e1, e2):
    e.spines[["top", "right"]].set_visible(False)
    e.legend(frameon=False)
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig(EVID / "tablero_hubs.png", dpi=130, bbox_inches="tight")
plt.show()

## 5. Catálogo para Athena y exportación para el tablero

- En la EC2 se lanza el *crawler* de Glue: registra o actualiza las tablas de gold para consultarlas con Athena
  (las consultas del tablero están en `tablero/consultas_athena.sql`).
- Además se dejan los KPI en CSV en `data/tablero/` (fuera de git) para herramientas que no se conectan a Athena.

In [ ]:
if BUCKET:
    import boto3

    glue = boto3.client("glue")
    nombre = cfg["catalogo"]["crawler"]
    try:
        glue.start_crawler(Name=nombre)
        print("Crawler lanzado:", nombre)
    except glue.exceptions.CrawlerRunningException:
        print("El crawler ya estaba corriendo:", nombre)
    time.sleep(30)   # el estado tarda unos segundos en pasar a RUNNING; sin esta pausa el bucle podría salir antes de empezar
    while glue.get_crawler(Name=nombre)["Crawler"]["State"] != "READY":
        time.sleep(20)
    print("Catálogo actualizado:", glue.get_crawler(Name=nombre)["Crawler"].get("LastCrawl", {}).get("Status"))
else:
    print("En local no hay catálogo de Glue: se omite.")

EXPORT = RAIZ / "data" / "tablero"
EXPORT.mkdir(parents=True, exist_ok=True)
spark.read.parquet(ruta("gold_kpi_diario")).toPandas().to_csv(EXPORT / "kpi_diario.csv", index=False)
kpi_mensual.to_csv(EXPORT / "kpi_alerta_mensual.csv", index=False)
for dim in ("gold_dim_aeropuerto", "gold_dim_aerolinea", "gold_dim_condicion_clima"):
    spark.read.parquet(ruta(dim)).toPandas().to_csv(EXPORT / f"{dim.removeprefix('gold_')}.csv", index=False)
print("CSV del tablero en", EXPORT)

In [ ]:
spark.stop()